# Spark Structured Streaming from Kafka with LakeLogic

The same broker, contract, events and checks as `kafka_streaming.ipynb`, but **Spark reads Kafka itself**:
`spark.readStream` pulls the topic, Spark's checkpoint folder tracks the offsets, and every micro-batch runs the
LakeLogic contract inside `foreachBatch` (`SparkStreamSink`). This is the path for Databricks and Fabric.

`kafka_json_stream(spark, topic, fields, brokers=...)` turns raw Kafka messages into rows for the contract:
- each message must be a JSON object; the contract's fields come out as text, so the engine's type checks decide;
- a message that is not a JSON object is **quarantined with the reason** (partition, offset, first bytes),
  where Spark's own `from_json` would quietly give a row of nulls;
- Kafka's timestamp, partition and offset are kept for ordering updates, then dropped.

Needs PySpark 3.5 with `delta-spark`; Spark downloads its Kafka connector on first run.

In [1]:
import os
import sys
os.environ.setdefault("KAFKA_BROKERS", "localhost:19092")
from loguru import logger
import polars as pl

logger.remove()
pl.Config.set_tbl_rows(12)
pl.Config.set_fmt_str_lengths(90)

import structured_scenarios as SS   # sets ENGINE=spark; reuses kafka_scenarios (K)
K = SS.K
spark = SS.start_spark()
print("Spark", spark.version)

Spark 3.5.9


## Run it from the contract file

The **same** [`contracts/rides_kafka.yaml`](contracts/rides_kafka.yaml) as the Polars/DuckDB notebook. On the Spark
engine `run_source()` uses Structured Streaming (`readStream` + `foreachBatch`), and Spark keeps its own offsets
in a checkpoint folder beside the one named in the contract (`rides_kafka_spark/`).

In [2]:
from pathlib import Path
print(Path("contracts/rides_kafka.yaml").read_text())

# Bronze rides from a Kafka topic. Everything the run needs is here: the broker, the topic,
# where the offsets are kept, the schema, the rules, and where good and bad rows go.
version: 1.0.0
dataset: rides
info: {title: bronze_rides_kafka}
primary_key: [ride_id]
source:
  type: stream
  options:
    kind: kafka
    brokers: env:KAFKA_BROKERS              # e.g. localhost:19092 (Redpanda in Docker)
    topic: rides
    starting_offsets: earliest
    checkpoint: data/checkpoints/rides_kafka.sqlite   # Polars/DuckDB offsets; Spark keeps its own in rides_kafka_spark/
    batch_size: 100
    trigger: available_now                  # drain to the current end, then stop; `continuous` keeps running
    # A secured cluster (SASL): the password is always an environment variable, never a literal.
    # security_protocol: SASL_SSL
    # sasl_mechanism: PLAIN
    # sasl_username: my-user
    # sasl_password: env:KAFKA_PASSWORD
model:
  fields:
    - {name: ride_id, type: long, required: true}
    

In [3]:
import shutil
from lakelogic import DataProcessor

for d in ("data/bronze", "data/quarantine", "data/checkpoints"):
    shutil.rmtree(d, ignore_errors=True)
K.reset_topic("rides")
K.produce("rides", [K.ride(i) for i in range(100)]
                   + [K.ride(900 + i, fare=-1.0) for i in range(5)]
                   + [b"not json"])

def run_contract(path="contracts/rides_kafka.yaml"):
    s = DataProcessor(engine="spark", contract=path).run_source().stream_summary
    print(f"read {s['source_count']} events in {s['batches']} micro-batch(es): "
          f"{s['good_count']} good, {s['bad_count']} quarantined")

run_contract()                                       # 100 good; 5 negative fares + 1 non-JSON quarantined
K.produce("rides", [K.ride(i) for i in range(200, 210)])
run_contract()                                       # only the 10 new events
run_contract()                                       # nothing new

read 106 events in 1 micro-batch(es): 100 good, 6 quarantined


read 10 events in 1 micro-batch(es): 10 good, 0 quarantined
read 0 events in 0 micro-batch(es): 0 good, 0 quarantined


## Under the hood: the hard cases, step by step

## 1. Drain the topic with Structured Streaming

The `available_now` trigger reads everything in the topic, in micro-batches, then stops.

In [4]:
r = K.Run("nb_structured")
r.send([K.ride(i) for i in range(500)]
       + [K.ride(1000 + i, fare=-5.0) for i in range(20)]
       + [{"city": "Rome", "fare": 9.0} for _ in range(10)])
r.send([b"{not json"], raw=True)

s = SS.structured(r)
print(f"read {s.source_count} in {s.batches} micro-batch(es): {s.good_count} good, {s.bad_count} quarantined")
display(r.table().sort("ride_id").head(5))
print("quarantine reasons (first of each kind):")
for reason in sorted({x.split(' (')[0] for x in r.quarantine_reasons()}):
    print(" -", reason)

read 531 in 1 micro-batch(es): 500 good, 31 quarantined


ride_id,city,fare,status
i64,str,f64,str
0,"""London""",5.0,"""completed"""
1,"""Paris""",6.0,"""completed"""
2,"""Lagos""",7.0,"""completed"""
3,"""London""",8.0,"""completed"""
4,"""Paris""",9.0,"""completed"""


quarantine reasons (first of each kind):
 - Message is not valid JSON
 - [pre] Rule failed: city_required
 - [pre] Rule failed: fare_not_negative
 - [pre] Rule failed: fare_required
 - [pre] Rule failed: ride_id_required


## 2. Run again: Spark's checkpoint reads only the new events

In [5]:
r.send([K.ride(i) for i in range(2000, 2060)])
again = SS.structured(r)
print("second run read", again.source_count, "events; bronze rows:", r.table().height)

second run read 60 events; bronze rows: 560


## 3. A crash inside a micro-batch

The second micro-batch fails after its write. Spark does not commit it, so the next run reads it again,
and the `merge` keeps one row per ride.

In [6]:
c = K.Run("nb_structured_crash")
c.send([K.ride(i) for i in range(300)])
crashed = SS.structured(c, per_batch=100, fail_in_batch=(2, "after"))
rest = SS.structured(c, per_batch=100)
t = c.table()
print(f"committed before the crash {crashed.source_count}, read after restart {rest.source_count}")
print("rows in bronze", t.height, "| one row per ride:", K.table_has_no_duplicates(t))

committed before the crash 99, read after restart 201
rows in bronze 300 | one row per ride: True


## 4. The same ride updated twice in one micro-batch: the last event wins

In [7]:
u = K.Run("nb_structured_updates", partitions=1)
u.send([K.ride(i) for i in range(5)] + [K.ride(i, status="refunded", fare=0.0) for i in range(2)])
su = SS.structured(u)
print(f"read {su.source_count}: good {su.good_count}, superseded {su.superseded_count}")
display(u.table().sort("ride_id"))

read 7: good 5, superseded 2


ride_id,city,fare,status
i64,str,f64,str
0,"""London""",0.0,"""refunded"""
1,"""Paris""",0.0,"""refunded"""
2,"""Lagos""",7.0,"""completed"""
3,"""London""",8.0,"""completed"""
4,"""Paris""",9.0,"""completed"""


## Scenario suite (Structured Streaming)

In [8]:
K.RESULTS.clear()
for name, fn, expect in SS.ALL:
    K.scenario(f"[structured] {name}", fn, **expect)
display(K.summary().select("status", "scenario", "result", "why"))

[PASS] [structured] drain: 500 good, 20 negative fares, 10 missing ride_id consumed=530, good=500, quarantined=30, rows_in_bronze=500, rows_in_quarantine=30


[PASS] [structured] resume: a later run reads only the new events first_run=200, second_run=60, third_run=0, rows_in_bronze=260


[PASS] [structured] crash inside batch 2, before its write; then resume committed_plus_reread=300, rows_in_bronze=300, no_duplicates=True


[PASS] [structured] crash inside batch 2, after its write; then resume committed_plus_reread=300, rows_in_bronze=300, no_duplicates=True


[PASS] [structured] replay the whole topic: merge keeps one row per ride replayed=150, rows_in_bronze=150, no_duplicates=True


[PASS] [structured] two messages are not JSON                     consumed=102, good=100, quarantined=2, rows_in_bronze=100, rows_in_quarantine=2, reason_says_not_json=True


[PASS] [structured] wrong type in a field; a new field appears    good=45, quarantined=5, rows_in_bronze=45


[PASS] [structured] the same ride updated later in the same batch consumed=40, superseded=10, rows_in_bronze=30, refunded=10, no_duplicates=True

8 of 8 streaming scenarios behaved as expected (spark).


status,scenario,result,why
str,str,str,str
"""PASS""","""[structured] drain: 500 good, 20 negative fares, 10 missing ride_id""","""consumed=530, good=500, quarantined=30, rows_in_bronze=500, rows_in_quarantine=30""",""""""
"""PASS""","""[structured] resume: a later run reads only the new events""","""first_run=200, second_run=60, third_run=0, rows_in_bronze=260""",""""""
"""PASS""","""[structured] crash inside batch 2, before its write; then resume""","""committed_plus_reread=300, rows_in_bronze=300, no_duplicates=True""",""""""
"""PASS""","""[structured] crash inside batch 2, after its write; then resume""","""committed_plus_reread=300, rows_in_bronze=300, no_duplicates=True""",""""""
"""PASS""","""[structured] replay the whole topic: merge keeps one row per ride""","""replayed=150, rows_in_bronze=150, no_duplicates=True""",""""""
"""PASS""","""[structured] two messages are not JSON""","""consumed=102, good=100, quarantined=2, rows_in_bronze=100, rows_in_quarantine=2, reason_sa…",""""""
"""PASS""","""[structured] wrong type in a field; a new field appears""","""good=45, quarantined=5, rows_in_bronze=45""",""""""
"""PASS""","""[structured] the same ride updated later in the same batch""","""consumed=40, superseded=10, rows_in_bronze=30, refunded=10, no_duplicates=True""",""""""
